<a href="https://colab.research.google.com/github/Ayushman2005-cmyk/DMPM_EXPERIMENTS/blob/main/DMPM_LAB6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

df = pd.read_csv('/content/Heart Attack.csv', header=0)

print("Dataset head:")
display(df.head())
print(f"Dataset shape: {df.shape}")

Dataset head:


,age,gender,impluse,pressurehight,pressurelow,glucose,kcm,troponin,class
0,64,1,66,160,83,160.0,1.80,0.012,negative
1,21,1,94,98,46,296.0,6.75,1.060,positive
2,55,1,64,160,77,270.0,1.99,0.003,negative
3,64,1,70,120,55,270.0,13.87,0.122,positive
4,55,1,64,112,65,300.0,1.08,0.003,negative


Dataset shape: (1319, 9)


In [36]:
df

,age,gender,impluse,pressurehight,pressurelow,glucose,kcm,troponin,class
0,64,1,66,160,83,160.0,1.80,0.012,negative
1,21,1,94,98,46,296.0,6.75,1.060,positive
2,55,1,64,160,77,270.0,1.99,0.003,negative
3,64,1,70,120,55,270.0,13.87,0.122,positive
4,55,1,64,112,65,300.0,1.08,0.003,negative
...,...,...,...,...,...,...,...,...,...
1314,44,1,94,122,67,204.0,1.63,0.006,negative
1315,66,1,84,125,55,149.0,1.33,0.172,positive
1316,45,1,85,168,104,96.0,1.24,4.250,positive
1317,54,1,58,117,68,443.0,5.80,0.359,positive


In [37]:
target_col = df.columns[-1]
X = df.drop(columns=[target_col])
y = df[target_col]

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce')
X = X.fillna(X.mean())

if y.dtype == 'object':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=y.name)

print("Target variable class distribution:")
display(y.value_counts())

if (y.value_counts() >= 2).all():
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
else:
    print("Warning: Some classes in the target variable have fewer than 2 samples. Splitting without stratification to avoid ValueError.")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Test target shape: {y_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully.")

Target variable class distribution:


,count
class,
1,810
0,509


Training features shape: (1055, 8)
Test features shape: (264, 8)
Training target shape: (1055,)
Test target shape: (264,)
Features scaled successfully.


In [38]:
from sklearn.ensemble import IsolationForest

print("Detecting outliers using Isolation Forest...")

iso_forest = IsolationForest(random_state=42, contamination=0.01)

outliers = iso_forest.fit_predict(X_train_scaled)

outlier_indices = (outliers == -1)

print(f"Number of outliers detected: {sum(outlier_indices)}")

X_train_cleaned = X_train_scaled[~outlier_indices]
y_train_cleaned = y_train[~outlier_indices]

print(f"Original training data shape: {X_train_scaled.shape}")
print(f"Cleaned training data shape: {X_train_cleaned.shape}")

X_train_for_knn = X_train_cleaned
y_train_for_knn = y_train_cleaned

Detecting outliers using Isolation Forest...
Number of outliers detected: 11
Original training data shape: (1055, 8)
Cleaned training data shape: (1044, 8)


In [39]:
knn = KNeighborsClassifier(n_neighbors=5)

print("Training KNN model...")
knn.fit(X_train_scaled, y_train)
print("KNN model training complete.")

y_pred = knn.predict(X_test_scaled)

print("Predictions made.")

Training KNN model...
KNN model training complete.
Predictions made.


In [40]:
knn_cleaned = KNeighborsClassifier(n_neighbors=5)

print("Training KNN model with cleaned data...")
knn_cleaned.fit(X_train_for_knn, y_train_for_knn)
print("KNN model training with cleaned data complete.")

y_pred_cleaned = knn_cleaned.predict(X_test_scaled)

print("Predictions made with cleaned data model.")

Training KNN model with cleaned data...
KNN model training with cleaned data complete.
Predictions made with cleaned data model.


In [41]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.6667
Precision: 0.6784
Recall: 0.6667
F1 Score: 0.6701

Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.65      0.60       102
           1       0.75      0.68      0.71       162

    accuracy                           0.67       264
   macro avg       0.66      0.66      0.66       264
weighted avg       0.68      0.67      0.67       264



In [42]:
accuracy_cleaned = accuracy_score(y_test, y_pred_cleaned)
precision_cleaned = precision_score(y_test, y_pred_cleaned, average='weighted', zero_division=0)
recall_cleaned = recall_score(y_test, y_pred_cleaned, average='weighted', zero_division=0)
f1_cleaned = f1_score(y_test, y_pred_cleaned, average='weighted', zero_division=0)

print("--- Performance after Outlier Removal ---")
print(f"Accuracy (Cleaned Data): {accuracy_cleaned:.4f}")
print(f"Precision (Cleaned Data): {precision_cleaned:.4f}")
print(f"Recall (Cleaned Data): {recall_cleaned:.4f}")
print(f"F1 Score (Cleaned Data): {f1_cleaned:.4f}")

print("\nClassification Report (Cleaned Data):")
print(classification_report(y_test, y_pred_cleaned, zero_division=0))

--- Performance after Outlier Removal ---
Accuracy (Cleaned Data): 0.6667
Precision (Cleaned Data): 0.6784
Recall (Cleaned Data): 0.6667
F1 Score (Cleaned Data): 0.6701

Classification Report (Cleaned Data):
              precision    recall  f1-score   support

           0       0.56      0.65      0.60       102
           1       0.75      0.68      0.71       162

    accuracy                           0.67       264
   macro avg       0.66      0.66      0.66       264
weighted avg       0.68      0.67      0.67       264

